In [4]:
import glob
import pandas as pd
import numpy as np

# 1) Read & concat
files = glob.glob("ul_tables_fixed_ep/UL_*.csv")
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# 2) Quantize Mass to exactly 0.0001‐GeV steps
df["Mass"] = df["Mass"].round(4)

# 3) (Optionally) ensure you have *every* 0.0001 step in the final table:
min_mass, max_mass = 0.033, 0.179
grid = np.round(np.arange(min_mass, max_mass + 1e-6, 0.0001), 4)
grid_df = pd.DataFrame({"Mass": grid})

# 4) Group & keep minima
combined = (
    df
    .groupby("Mass", as_index=False)
    .agg({
        "Upper Limit":      "min",
        "Epsilon Squared":  "min"
    })
)

# 5) Merge onto the full grid (so you get *all* 0.0001 points—even if some never appeared,
#    those will come in with NaN or you can fill with np.inf)
final = grid_df.merge(combined, on="Mass", how="left")

# 6) Fill any missing points with inf (optional)
final["Upper Limit"].fillna(np.inf, inplace=True)
final["Epsilon Squared"].fillna(np.inf, inplace=True)

# 7) Write it out
final.to_csv("combined_UL.csv", index=False)
print(f"Wrote {len(final)} rows → combined_UL.csv")


Wrote 1461 rows → combined_UL.csv


/var/folders/lh/5k1xk47s7q9dgdbf5kby_6_h0000gn/T/ipykernel_7874/325257080.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final["Upper Limit"].fillna(np.inf, inplace=True)
/var/folders/lh/5k1xk47s7q9dgdbf5kby_6_h0000gn/T/ipykernel_7874/325257080.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alway

In [ ]:
combined = pd.read_csv("combined_UL_brazil_range.csv")
h = combined.groupby('Mass', as_index=False).min()

# keep only 0.033 ≤ Mass ≤ 0.179
mask = (combined['Mass'] >= 0.033) & (combined['Mass'] <= 0.179)
h = combined.loc[mask]

h.to_csv('combined_UL_brazil_range.csv', index=False)
